In [1]:
from datasets import Dataset
from datetime import datetime, timedelta
import os
import re
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from time import sleep
import torch
from transformers import pipeline

### GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS 
### GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS 
### GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS | GLOBAL VARS 

# this basically means "smoke em if you got em" where the "em" is NVIDIA GPU
DEVICE = 0 if torch.cuda.is_available() else -1

SF_USR = os.getenv('SF_USR')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')

# connect to database and init a cursor for querying
xct_params = {
    "user":                 SF_USR
   ,"account":              SF_ID
   ,"warehouse":            SF_WH
   ,"database":             SF_DB
   ,"schema":               SF_SC
   ,"role":                 SF_RL
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}

SF_XCT = snowflake.connector.connect(**xct_params) #connection object
CSR = SF_XCT.cursor()

In [2]:
query = f"""select * from {SF_DB}.{SF_SC}.text_keywords_summarystats"""
CSR.execute(query)
df = CSR.fetch_pandas_all()



In [3]:
#Import necessary libraries
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import GofChisquarePower, TTestIndPower, NormalIndPower
from scipy.stats import norm

In [6]:
# Definition of Parameters
EST_PROPORTION = 0.50
CONFIDENCE_LEVEL = 0.95
MARGIN_OF_ERROR = 0.05 
DESIRED_POWER = 0.8
ALPHA = 1 - CONFIDENCE_LEVEL
Z_SCORE = norm.ppf(1- ALPHA / 2)

REQ_SAMPLE_SIZE = np.ceil((Z_SCORE**2 * EST_PROPORTION * (1 - EST_PROPORTION)) / (MARGIN_OF_ERROR**2))


Statistical test: 
Cochran's Formula for Sample Size.

When population is very large compared to sample size.

In [ ]:
#iterate for each keyword
df['POST_COUNT_SAMPLE_SUFFICIENT'] = False
df['POS_SENTIMENT_SAMPLE_SUFFICIENT'] = False
df['NEU_SENTIMENT_SAMPLE_SUFFICIENT'] = False
df['NEG_SENTIMENT_SAMPLE_SUFFICIENT'] = False

sentiment_columns = ['POSITIVE_POSTS', 'NEUTRAL_POSTS', 'NEGATIVE_POSTS']
sufficiency_cols = ['POS_SENTIMENT_SAMPLE_SUFFICIENT', 'NEU_SENTIMENT_SAMPLE_SUFFICIENT', 'NEG_SENTIMENT_SAMPLE_SUFFICIENT']

for index, row in df.iterrows():
    # Check 1: Sufficiency of the total keyword sample
    if row['UNIQUE_POSTS'] >= REQ_SAMPLE_SIZE:
        df.loc[index, 'TOTAL_SAMPLE_SUFFICIENT'] = True
    
    # Check 2: Sufficiency of each sentiment subsample (e.g.sufficient positive posts for a keyword to infer about all posts with that keyword; considers each sentiment individually)
    for sentiment_col, sufficiency_col in zip(sentiment_columns, sufficiency_cols):
        if row[sentiment_col] >= REQ_SAMPLE_SIZE:
            df.loc[index, sufficiency_col] = True

print(df)

           KEYWORD             TOPIC        CATEGORY  UNIQUE_POSTS  \
0              ICE       US domestic  issue specific        106771   
1              TDS       culture war           coded           200   
2             acab       US domestic           coded           981   
3      afghanistan  global conflicts  issue specific           423   
4         allyship       culture war           coded            57   
..             ...               ...             ...           ...   
99         ukraine  global conflicts  issue specific          6721   
100      west bank                IP  issue specific           275   
101           woke       culture war           coded          2890   
102  working class          economic  issue specific           523   
103       zelensky  global conflicts  issue specific           970   

     POSITIVE_POSTS PCT_POSITIVE  NEGATIVE_POSTS PCT_NEGATIVE  NEUTRAL_POSTS  \
0             28454    26.649558           41041    38.438340          37276   